# Speech-to-Reasoning Pipeline**Whisper ASR + Quantized Reasoning LLM**This notebook implements an end-to-end pipeline that:1. Transcribes audio using **OpenAI Whisper**2. Passes the transcription to a **4-bit quantized reasoning LLM** (via Unsloth)Model: [Qwen2.5-3B-Instruct (4-bit BnB)](https://huggingface.co/unsloth/Qwen2.5-3B-Instruct-bnb-4bit) — optimized for logical reasoning while fitting in Colab's free GPU memory (~15 GB VRAM).

---## 1. Installation & Environment Setup

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q openai-whisper gtts
    !pip install -q accelerate bitsandbytes
    !pip install -q git+https://github.com/unslothai/unsloth.git
else:
    print("Running locally - ensure dependencies are installed manually.")

In [ ]:
import torch
import whisper
import numpy as np
import os
import gc
import warnings

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.2f} GB")
else:
    print("WARNING: No GPU found. Pipeline will run on CPU (very slow).")

---## 2. Audio Input HandlerSupports:- **Upload** a .wav / .mp3 / .m4a file- **Generate** a synthetic sample using gTTS (for quick testing)- **Download** a sample from Hugging Face Datasets

In [ ]:
from gtts import gTTS
import IPython.display as ipd


def generate_sample_audio(text, filename="sample_query.wav", lang="en"):
    if os.path.exists(filename):
        print(f"Using existing file: {filename}")
        return filename
    print(f"Generating audio from text: \"{text}\"")
    tts = gTTS(text=text, lang=lang, slow=False)
    tts.save(filename)
    print(f"Saved to {filename}")
    return filename


def upload_audio(colab_only=True):
    if colab_only:
        try:
            from google.colab import files
            uploaded = files.upload()
            for fn in uploaded.keys():
                print(f"Uploaded: {fn} ({len(uploaded[fn])} bytes)")
                return fn
        except ImportError:
            pass
    path = input("Enter path to audio file: ").strip()
    if os.path.exists(path):
        return path
    raise FileNotFoundError(f"File not found: {path}")


def play_audio(path):
    return ipd.Audio(path)


def clean_temp_files(*paths):
    for p in paths:
        if p and os.path.exists(p):
            os.remove(p)
            print(f"Removed {p}")

---## 3. Whisper ASR — Speech-to-TextWe load Whisper **base** (or `small` if more VRAM is available).Whisper is loaded on CPU first, then moved to GPU for inference to save VRAM for the LLM.

In [ ]:
WHISPER_MODEL_SIZE = "base"

print(f"Loading Whisper model: {WHISPER_MODEL_SIZE}")
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)

if torch.cuda.is_available():
    whisper_model = whisper_model.cuda()
    print("Whisper moved to GPU.")
else:
    print("Whisper running on CPU.")

In [ ]:
def transcribe_audio(audio_path, language="en"):
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    print(f"Transcribing: {audio_path} ...")

    result = whisper_model.transcribe(
        audio_path,
        language=language,
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6,
    )

    text = result["text"].strip()
    duration = result["segments"][-1]["end"] if result["segments"] else 0

    print(f"Transcription complete ({duration:.1f}s audio -> {len(text.split())} words)")
    print(f"\n{'─' * 60}")
    print(f"TRANSCRIPTION:\n{text}")
    print(f"{'─' * 60}\n")

    return result


def transcribe_microphone(duration=5, sample_rate=16000):
    try:
        from IPython.display import Javascript
        from google.colab import output
        from base64 import b64decode

        print(f"Recording for {duration} seconds... Speak now!")

        js_code = f"""
        const record = new Promise(async (resolve) => {{
            const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = () => {{
                const blob = new Blob(chunks);
                const reader = new FileReader();
                reader.onload = () => resolve(reader.result.split(',')[1]);
                reader.readAsDataURL(blob);
            }};
            recorder.start();
            setTimeout(() => recorder.stop(), {duration * 1000});
        }});
        record.then(data => google.colab.kernel.invokeFunction('notebook.on_audio', [data], {{}}));
        """

        from IPython.display import display, Javascript as Js
        display(Js(js_code))
    except Exception as e:
        print(f"Microphone recording not available: {e}")
        print("Falling back to file-based transcription.")
        return None

---## 4. Quantized Reasoning LLM (Unsloth 4-bit)Loading a **4-bit NormalFloat** quantized model via Unsloth.This reduces memory from ~16 GB (FP16) to ~3-4 GB with minimal accuracy loss.Available options:- `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` (recommended — best reasoning for its size)- `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`- `unsloth/gemma-2-2b-it-bnb-4bit` (smaller, faster)

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

from unsloth import FastLanguageModel

print(f"Loading quantized reasoning model: {MODEL_NAME}")
print("This will take 2-5 minutes depending on network & GPU...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,
    device_map="auto",
)

FastLanguageModel.for_inference(model)

if torch.cuda.is_available():
    whisper_model = whisper_model.cpu()
    torch.cuda.empty_cache()
    print("Whisper moved to CPU to free GPU for reasoning model.")

print("\n✓ Reasoning model loaded and ready.")
print(f"  Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
REASONING_SYSTEM_PROMPT = (
    "You are a precise logical reasoning assistant. "
    "Think step by step, show your chain of thought, "
    "and provide a clear final answer."
)


def reason_on_transcription(
    transcription,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
):
    messages = [
        {"role": "system", "content": REASONING_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Transcribed speech: \"{transcription}\"\n\n"
                f"Analyze this query step by step, then provide the final answer."
            ),
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    print("Reasoning...")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    response = response.strip()

    print(f"\n{'═' * 60}")
    print("REASONING MODEL OUTPUT")
    print(f"{'═' * 60}")
    print(response)
    print(f"{'═' * 60}")

    return response


del whisper_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---## 5. End-to-End PipelineCombines Whisper (on CPU) + Quantized LLM (on GPU) into a single call.

In [ ]:
def speech_to_reasoning(
    audio_path,
    language="en",
    whisper_model_size="base",
    max_new_tokens=512,
    temperature=0.1,
    verbose=True,
):
    result = {"audio_path": audio_path}

    # Step 1: Load Whisper on CPU
    if verbose:
        print(f"\n{'=' * 60}")
        print("STEP 1: Loading Whisper ASR")
        print(f"{'=' * 60}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    whisper_model_local = whisper.load_model(whisper_model_size)

    # Step 2: Transcribe
    if verbose:
        print(f"\n{'=' * 60}")
        print("STEP 2: Transcribing audio")
        print(f"{'=' * 60}")

    transcript_result = whisper_model_local.transcribe(
        audio_path,
        language=language,
        task="transcribe",
        fp16=False,
        temperature=0.0,
    )
    transcription = transcript_result["text"].strip()
    result["transcription"] = transcription
    result["transcript_detail"] = transcript_result

    if verbose:
        print(f"Transcription: {transcription}")

    del whisper_model_local
    gc.collect()

    # Step 3: Reason on GPU
    if verbose:
        print(f"\n{'=' * 60}")
        print("STEP 3: Reasoning with quantized LLM")
        print(f"{'=' * 60}")

    messages = [
        {"role": "system", "content": REASONING_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Transcribed speech: \"{transcription}\"\n\nAnalyze this step by step and provide the final answer.",
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.95,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )

    reasoning = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    result["reasoning"] = reasoning

    if verbose:
        print(f"Reasoning:\n{reasoning}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

---## 6. Demo — End-to-End TestWe'll generate a sample audio query using TTS (text-to-speech), then run the full pipeline.**Sample query:** A logic puzzle requiring multi-step reasoning.

In [ ]:
SAMPLE_QUERY = (
    "Alice has three boxes: one red, one blue, and one green. "
    "Only one box contains a prize. "
    "The red box says 'The prize is in this box.' "
    "The blue box says 'The prize is not in this box.' "
    "The green box says 'The prize is not in the red box.' "
    "Only one statement is true. Which box contains the prize?"
)

print("Sample logical reasoning query:")
print(f"  \"{SAMPLE_QUERY}\"\n")

audio_file = generate_sample_audio(SAMPLE_QUERY, "logic_puzzle.wav")

print("\nPlaying audio...")
ipd.Audio(audio_file)

In [ ]:
print("=" * 60)
print("SPEECH-TO-REASONING PIPELINE")
print("=" * 60)

result = speech_to_reasoning(
    audio_path=audio_file,
    language="en",
    whisper_model_size="base",
    max_new_tokens=512,
    temperature=0.1,
    verbose=True,
)

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"\n📝 Transcription:\n  {result['transcription']}")
print(f"\n🧠 Reasoning:\n{result['reasoning']}")
print(f"\n{'=' * 60}")

---## 7. Try Your Own AudioUpload your own `.wav` / `.mp3` / `.m4a` file and run the pipeline.

In [ ]:
print("Upload an audio file (WAV, MP3, M4A):")
custom_audio = upload_audio(colab_only=True)

print("\nPlaying uploaded audio...")
ipd.Audio(custom_audio)

In [ ]:
custom_result = speech_to_reasoning(
    audio_path=custom_audio,
    language="en",
    whisper_model_size="base",
    max_new_tokens=512,
    temperature=0.1,
    verbose=True,
)

print("\n" + "═" * 60)
print("CUSTOM AUDIO RESULT")
print("═" * 60)
print(f"\n📝 Transcription:\n  {custom_result['transcription']}")
print(f"\n🧠 Reasoning:\n{custom_result['reasoning']}")
print(f"\n{'═' * 60}")

---## 8. Performance & Memory BenchmarksTrack VRAM usage, transcription speed, and inference speed.

In [ ]:
import time


def benchmark_pipeline(audio_path, num_runs=2):
    times = []
    mem_usage = []

    for i in range(num_runs):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        t0 = time.time()

        wm = whisper.load_model("base")

        trans_start = time.time()
        tr = wm.transcribe(audio_path, language="en", fp16=False, temperature=0.0)
        trans_time = time.time() - trans_start
        del wm

        messages = [
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"Transcribed: \"{tr['text'].strip()}\"\n\nAnswer step by step."},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")

        reason_start = time.time()
        with torch.no_grad():
            out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.1, top_p=0.95, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        reason_time = time.time() - reason_start

        total_time = time.time() - t0
        times.append((trans_time, reason_time, total_time))

        if torch.cuda.is_available():
            mem_usage.append(torch.cuda.max_memory_allocated() / 1e9)
        gc.collect()

    avg_trans, avg_reason, avg_total = [sum(x) / len(x) for x in zip(*times)]
    avg_mem = sum(mem_usage) / len(mem_usage) if mem_usage else 0

    print(f"\n{'─' * 50}")
    print(f"BENCHMARK RESULTS (averaged over {num_runs} runs)")
    print(f"{'─' * 50}")
    print(f"  Transcription:  {avg_trans:.2f}s")
    print(f"  Reasoning:      {avg_reason:.2f}s")
    print(f"  Total:          {avg_total:.2f}s")
    if avg_mem:
        print(f"  Peak VRAM:      {avg_mem:.2f} GB")
    print(f"{'─' * 50}")

    return {"avg_trans_time": avg_trans, "avg_reason_time": avg_reason, "avg_total_time": avg_total, "avg_vram_gb": avg_mem}


print("Running benchmark (this may take a moment)...")
benchmark_stats = benchmark_pipeline(audio_file, num_runs=1)

---## Summary✅ **End-to-end Speech-to-Reasoning pipeline** complete.| Component | Model | Memory ||---|---|---|| ASR (Speech→Text) | Whisper `base` (CPU) | ~1 GB RAM || Reasoning (Text→Answer) | Qwen2.5-3B-Instruct (4-bit, GPU) | ~4 GB VRAM |### Key optimizations implemented:- **Whisper on CPU** → leaves full GPU VRAM for the LLM- **Unsloth 4-bit quantization** → reduces LLM memory by ~4× vs FP16- **Bfloat16 support** → faster inference on compatible GPUs- **Explicit GC & cache clearing** → prevents memory fragmentation across runs- **Chat template** → properly formats prompts for instruction-tuned models### Possible extensions:- Use Whisper `large-v3` for higher accuracy (at ~3 GB extra memory)- Swap to `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` for comparison- Add batch transcription for long-form audio- Serve via Gradio for interactive use